# import

In [845]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 데이터 로드 및 정보

In [846]:
train_path = r"..\datasets\train.csv"
test_path  = r"..\datasets\test.csv"

train_df = pd.read_csv(train_path, encoding="utf-8-sig")
test_df  = pd.read_csv(test_path,  encoding="utf-8-sig")

print("=== 데이터 기본 정보 ===")
print(f"train shape: {train_df.shape}")
print(f"test shape: {test_df.shape}")
print(f"\n원본 데이터 피처 개수(타깃 포함): {len(train_df.columns)}개")
print("\n[종속변수 분포]")
print(train_df['completed'].value_counts())

=== 데이터 기본 정보 ===
train shape: (748, 46)
test shape: (814, 45)

원본 데이터 피처 개수(타깃 포함): 46개

[종속변수 분포]
completed
0    525
1    223
Name: count, dtype: int64


# 데이터 분할

테스트 세트가 별도의 파일(test.csv)로 존재하기 때문에, 학습 세트(train.csv)에서
'검증 세트'만 별도로 분할했습니다.

In [847]:

X = train_df.drop(columns=['completed'])
y = train_df['completed']

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("=== 분할 결과 ===")
print(f"train: {X_train.shape}, valid: {X_valid.shape}")
print(f"\n[학습 세트] 종속변수 분포:\n{y_train.value_counts()}")
print(f"\n[검증 세트] 종속변수 분포:\n{y_valid.value_counts()}")


=== 분할 결과 ===
train: (598, 45), valid: (150, 45)

[학습 세트] 종속변수 분포:
completed
0    420
1    178
Name: count, dtype: int64

[검증 세트] 종속변수 분포:
completed
0    105
1     45
Name: count, dtype: int64


# 데이터 전처리 - 변수 제거(1)

[ID] -> 고유 식별자

[generation], [nationality] -> 상수

[school1] -> 카테고리 개수 너무 많고, 변수 의미상 희소 카테고리를 '기타'로 합치기 애매하다고 판단(실제 인적사항이어서)

[interested_company], [incumbents_lecture_scale_reason] -> 서술형, 모델링 이후 과정에서 drop 여부를 다시 고려

In [848]:

drop_cols = [
    'ID', 
    'generation', 'nationality', 
    'school1', 
    'interested_company', 'incumbents_lecture_scale_reason'
]

train_clean = X_train.drop(columns=[col for col in drop_cols if col in X_train.columns])
valid_clean = X_valid.drop(columns=[col for col in drop_cols if col in X_valid.columns])
test_clean = test_df.drop(columns=[col for col in drop_cols if col in test_df.columns])

print(f"제거 전: {len(X_train.columns)} → 제거 후: {len(train_clean.columns)}")
print("제거된:", [col for col in drop_cols if col in X_train.columns])


제거 전: 45 → 제거 후: 39
제거된: ['ID', 'generation', 'nationality', 'school1', 'interested_company', 'incumbents_lecture_scale_reason']


결측치 개수가 전체 데이터의 50% 이상인 12개의 피처 제거

In [849]:

def drop_high_missing(df_list, threshold=0.5):
    common_cols = set.intersection(*[set(df.columns) for df in df_list])
    na_ratios = {}
    
    for col in common_cols:
        na_cnt = sum(df[col].isna().sum() for df in df_list)
        na_ratio = na_cnt / sum(len(df) for df in df_list)
        na_ratios[col] = na_ratio
    
    high_na_cols = [col for col, ratio in na_ratios.items() if ratio >= threshold]
    
    # 제거 후 데이터프레임 반환
    cleaned_dfs = []
    for df in df_list:
        cleaned = df.drop(columns=[col for col in high_na_cols if col in df.columns])
        cleaned_dfs.append(cleaned)
    
    print(f"결측치 {threshold*100}%↑ 제거: {len(high_na_cols)}개")
    print("제거된 컬럼:", sorted(high_na_cols))
    return cleaned_dfs

# 실행
train_clean, valid_clean, test_clean = drop_high_missing(
    [train_clean, valid_clean, test_clean]
)

print(f"\n최종 shape: train={train_clean.shape}, valid={valid_clean.shape}, test={test_clean.shape}")


결측치 50.0%↑ 제거: 12개
제거된 컬럼: ['class2', 'class3', 'class4', 'contest_award', 'contest_participation', 'idea_contest', 'previous_class_3', 'previous_class_4', 'previous_class_5', 'previous_class_6', 'previous_class_7', 'previous_class_8']

최종 shape: train=(598, 27), valid=(150, 27), test=(814, 27)


# 데이터 전처리 - 결측치/이상치 처리(2)

1. 결측치 처리

연속형 변수

[completed_semester] -> 중앙값으로 대체 후 정수형으로 변환 


범주형 변수

[major1_1] → ‘Unknown’으로 대체

[major type] → major1_1, major1_2 값이 둘 다 존재한다면 '복수 전공 ( 다중전공, 이중전공 포함 )', 그렇지 않다면 '단일 전공'으로 대체

[major_field] → ‘Unknown’으로 대체

[major1_2] → ‘없음’으로 대체

In [850]:
# 여섯 번째 셀: 결측치 처리

def print_missing_combined(train_df, valid_df, stage):
    cols = ['major1_2','completed_semester', 'major_field', 'major type', 'major1_1']
    total_missing = {}
    
    for col in cols:
        if col in train_df.columns:
            total_missing[col] = (train_df[col].isnull().sum() + 
                                valid_df[col].isnull().sum())
    
    missing = pd.Series(total_missing).sort_values(ascending=False)
    print(f"=== {stage} 결측치 (학습+검증 합계) ===")
    print(missing)
    print()

# 처리 전
print_missing_combined(train_clean, valid_clean, "처리 전")

# 1. completed_semester
median_sem = train_clean['completed_semester'].median()
for df in [train_clean, valid_clean, test_clean]:
    df['completed_semester'] = df['completed_semester'].fillna(median_sem).astype(int)

# 2. 범주형
for df in [train_clean, valid_clean, test_clean]:
    df['major1_1'] = df['major1_1'].fillna('Unknown')
    df['major1_2'] = df['major1_2'].fillna('없음')
    df['major_field'] = df['major_field'].fillna('Unknown')
    if 'major type' in df.columns:
        df['major type'] = df['major type'].fillna('단일 전공')

# 3. major_type 생성
def create_major_type(row):
    m1 = pd.notna(row.get('major1_1'))
    m2 = pd.notna(row.get('major1_2')) and row['major1_2'] != '없음' # major1_2가 공백이 아님과 동시에 '없음'이 아니면 복수 전공으로 간주
    return '복수 전공 ( 다중전공, 이중전공 포함 )' if m1 and m2 else '단일 전공'

for df in [train_clean, valid_clean, test_clean]:
    df['major_type'] = df.apply(create_major_type, axis=1)

# 처리 후
print_missing_combined(train_clean, valid_clean, "처리 후")


=== 처리 전 결측치 (학습+검증 합계) ===
major1_2              439
completed_semester     28
major_field            23
major type             22
major1_1               20
dtype: int64

=== 처리 후 결측치 (학습+검증 합계) ===
major1_2              0
completed_semester    0
major_field           0
major type            0
major1_1              0
dtype: int64



2. 이상치 처리

[completed_semester] 

-> 중앙값으로 대체. 

값이 0인 데이터가 하나 있었지만, ‘job’==‘직장인’, ‘major1_1’==‘기타’, ‘school1’==0인 것으로 보아, 대학 진학없이 바로 취직한 특이 케이스로 간주하여 이상치 처리 X

In [851]:
def print_outlier_status(train_df, valid_df, stage):
    col = 'completed_semester'
    outliers_train = train_df[col] > 100
    outliers_valid = valid_df[col] > 100
    total = outliers_train.sum() + outliers_valid.sum()
    
    print(f"=== {stage} 이상치 (학습+검증) ===")
    print(f"총 이상치: {total}개")
    print(f"학습: {outliers_train.sum()}개 {train_df.loc[outliers_train, col].values}")
    print(f"검증: {outliers_valid.sum()}개 {valid_df.loc[outliers_valid, col].values}")
    print()

print_outlier_status(train_clean, valid_clean, "처리 전")

# 범위로 안전 처리
median_clean = train_clean['completed_semester'].median()
for df in [train_clean, valid_clean]:
    df.loc[df['completed_semester'] > 100, 'completed_semester'] = median_clean

test_clean['completed_semester'] = test_clean['completed_semester'].clip(0, 20)

print_outlier_status(train_clean, valid_clean, "처리 후")


=== 처리 전 이상치 (학습+검증) ===
총 이상치: 3개
학습: 2개 [20241  2020]
검증: 1개 [20241]

=== 처리 후 이상치 (학습+검증) ===
총 이상치: 0개
학습: 0개 []
검증: 0개 []



# 데이터 전처리 - 범주형 변수 인코딩(3)

[major type], [re_registration], [project_type], [major_data], [incumbents_level]

-> 이진값(카테고리가 2개)을 가지는 5개의 변수: 레이블 인코딩

In [852]:
# 8번째 셀: 이진 범주형 레이블 인코딩 (수정)

binary_features = {
    'major type': ['단일 전공', '복수 전공 ( 다중전공, 이중전공 포함 )'],
    're_registration': ['예', '아니요'],
    'project_type': ['개인', '팀'],
    'major_data': ['TRUE', 'FALSE'],  # 문자열로 통일
    'incumbents_level': ['주니어 (0~3년차)', '시니어 (10년차 ~)']
}

print("=== 매핑 기준 (0=첫번째, 1=두번째) ===")
for col, mapping in binary_features.items():
    print(f"{col:15s} | 0={mapping[0]}, 1={mapping[1]}")


for col, mapping in binary_features.items():
    label_map = {mapping[0]: 0, mapping[1]: 1}
    
    for df in [train_clean, valid_clean, test_clean]:
        df[f'{col}_enc'] = df[col].map(label_map)
    
    for df in [train_clean, valid_clean, test_clean]:
        df.drop(col, axis=1, inplace=True)

print("\n✓ 이진 인코딩 완료")


=== 매핑 기준 (0=첫번째, 1=두번째) ===
major type      | 0=단일 전공, 1=복수 전공 ( 다중전공, 이중전공 포함 )
re_registration | 0=예, 1=아니요
project_type    | 0=개인, 1=팀
major_data      | 0=TRUE, 1=FALSE
incumbents_level | 0=주니어 (0~3년차), 1=시니어 (10년차 ~)

✓ 이진 인코딩 완료


[certificate_acquisition], [desired_certificate], [onedayclass_topic] 

-> 데이터 수가 3 이하인 희소 카테고리들을 모두 '희소'로 통합

그 후 멀티-핫 인코딩(다중 선택이 가능하므로)

In [853]:
# 9-1 셀: certificate_acquisition 멀티-핫 (데이터 백업)

col = 'certificate_acquisition'

# 0. 백업 (원본 보존)
orig_col = f'{col}_orig'
train_clean[orig_col] = train_clean[col]
valid_clean[orig_col] = valid_clean[col]
test_clean[orig_col] = test_clean[col]

# 1. 처리 전 (백업 데이터)
categories = []
for val in train_clean[orig_col].dropna():
    cats = [c.strip() for c in str(val).split(',')]
    categories.extend(cats)

before_count = Counter(categories)
print(f"\n=== {col} 처리 전 (13종) ===")
for cat, cnt in sorted(before_count.items(), key=lambda x: x[1], reverse=True):
    print(f"  {cat:30s} {cnt:4d}")

# 2. 희소 통합
rare_cats = {cat for cat, cnt in before_count.items() if cnt <= 3}
after_cats = sorted((set(before_count) - rare_cats) | {'희소'})

print(f"\n=== {col} 처리 후 ({len(after_cats)}종) ===")
for cat in sorted(after_cats):
    new_cnt = sum(1 for cats in train_clean[orig_col].dropna() 
                  for c in [cc.strip() for cc in str(cats).split(',')] 
                  if (c == cat) or (c in rare_cats and cat == '희소'))
    print(f"  {cat:30s} {new_cnt:4d}")

# 3. 인코딩
new_cols = [f'{col}_{c}' for c in after_cats]
for df, orig in zip([train_clean, valid_clean, test_clean], 
                   [train_clean[orig_col], valid_clean[orig_col], test_clean[orig_col]]):
    def encode(val):
        if pd.isna(val): return pd.Series({c:0 for c in after_cats})
        cats = [c.strip() for c in str(val).split(',')]
        result = {c:0 for c in after_cats}
        for c in cats:
            key = '희소' if c in rare_cats else c
            result[key] = 1
        return pd.Series(result)
    
    df[new_cols] = orig.apply(encode)[after_cats].values

print(f"\n✓ {len(after_cats)}개 피처 완료")



=== certificate_acquisition 처리 전 (13종) ===
  없음                              370
  ADsP                            158
  SQLD                             96
  구글 애널리스트                         18
  정보처리기사                           16
  빅데이터 분석 기사                        9
  컴퓨터활용능력                           3
  기타                                3
  준비중: SQLD                         2
  준비중                               2
  준비중: ADsP                         1
  AWS                               1
  태블로                               1

=== certificate_acquisition 처리 후 (7종) ===
  ADsP                            158
  SQLD                             96
  구글 애널리스트                         18
  빅데이터 분석 기사                        9
  없음                              370
  정보처리기사                           16
  희소                               13

✓ 7개 피처 완료


In [854]:
# 9-2 셀: desired_certificate 멀티-핫 (Top10 출력)

col = 'desired_certificate'

# 0. 백업
orig_col = f'{col}_orig'
train_clean[orig_col] = train_clean[col]
valid_clean[orig_col] = valid_clean[col]
test_clean[orig_col] = test_clean[col]

# 1. 처리 전 (Top10)
categories = []
for val in train_clean[orig_col].dropna():
    cats = [c.strip() for c in str(val).split(',')]
    categories.extend(cats)

before_count = Counter(categories)
print(f"\n=== {col} 처리 전 ({len(before_count)}종) ===")
print("Top 10:")
for cat, cnt in before_count.most_common(10):
    print(f"  {cat:30s} {cnt:4d}")
if len(before_count) > 10:
    print(f"  ... +{len(before_count)-10}개")

# 2. 희소 통합
rare_cats = {cat for cat, cnt in before_count.items() if cnt <= 3}
after_cats = sorted((set(before_count) - rare_cats) | {'희소'})

print(f"\n=== {col} 처리 후 ({len(after_cats)}종) ===")
print("Top 10:")
after_count = Counter()
for val in train_clean[orig_col].dropna():
    cats = [c.strip() for c in str(val).split(',')]
    for c in cats:
        key = '희소' if c in rare_cats else c
        after_count[key] += 1

for cat, cnt in after_count.most_common(10):
    print(f"  {cat:30s} {cnt:4d}")

# 3. 인코딩
new_cols = [f'{col}_{c}' for c in after_cats]
for df, orig in zip([train_clean, valid_clean, test_clean], 
                   [train_clean[orig_col], valid_clean[orig_col], test_clean[orig_col]]):
    def encode(val):
        if pd.isna(val): return pd.Series({c:0 for c in after_cats})
        cats = [c.strip() for c in str(val).split(',')]
        result = {c:0 for c in after_cats}
        for c in cats:
            key = '희소' if c in rare_cats else c
            result[key] = 1
        return pd.Series(result)
    
    df[new_cols] = orig.apply(encode)[after_cats].values

print(f"\n✓ {len(after_cats)}개 피처 완료")



=== desired_certificate 처리 전 (37종) ===
Top 10:
  빅데이터 분석 기사                      368
  SQLD                            342
  ADsP                            273
  정보처리기사                          225
  구글 애널리스트                        121
  태블로 관련 자격증                      113
  .                                 3
  ADP                               2
  사회조사분석사                           2
  SQLP                              2
  ... +27개

=== desired_certificate 처리 후 (7종) ===
Top 10:
  빅데이터 분석 기사                      368
  SQLD                            342
  ADsP                            273
  정보처리기사                          225
  구글 애널리스트                        121
  태블로 관련 자격증                      113
  희소                               36

✓ 7개 피처 완료


In [855]:
# 9-3 셀: onedayclass_topic 멀티-핫 (Top10 출력)

col = 'onedayclass_topic'

# 0. 백업
orig_col = f'{col}_orig'
train_clean[orig_col] = train_clean[col]
valid_clean[orig_col] = valid_clean[col]
test_clean[orig_col] = test_clean[col]

# 1. 처리 전 (Top10)
categories = []
for val in train_clean[orig_col].dropna():
    cats = [c.strip() for c in str(val).split(',')]
    categories.extend(cats)

before_count = Counter(categories)
print(f"\n=== {col} 처리 전 ({len(before_count)}종) ===")
print("Top 10:")
for cat, cnt in before_count.most_common(10):
    print(f"  {cat:30s} {cnt:4d}")
if len(before_count) > 10:
    print(f"  ... +{len(before_count)-10}개")

# 2. 희소 통합
rare_cats = {cat for cat, cnt in before_count.items() if cnt <= 3}
after_cats = sorted((set(before_count) - rare_cats) | {'희소'})

print(f"\n=== {col} 처리 후 ({len(after_cats)}종) ===")
print("Top 10:")
after_count = Counter()
for val in train_clean[orig_col].dropna():
    cats = [c.strip() for c in str(val).split(',')]
    for c in cats:
        key = '희소' if c in rare_cats else c
        after_count[key] += 1

for cat, cnt in after_count.most_common(10):
    print(f"  {cat:30s} {cnt:4d}")

# 3. 인코딩
new_cols = [f'{col}_{c}' for c in after_cats]
for df, orig in zip([train_clean, valid_clean, test_clean], 
                   [train_clean[orig_col], valid_clean[orig_col], test_clean[orig_col]]):
    def encode(val):
        if pd.isna(val): return pd.Series({c:0 for c in after_cats})
        cats = [c.strip() for c in str(val).split(',')]
        result = {c:0 for c in after_cats}
        for c in cats:
            key = '희소' if c in rare_cats else c
            result[key] = 1
        return pd.Series(result)
    
    df[new_cols] = orig.apply(encode)[after_cats].values

print(f"\n✓ {len(after_cats)}개 피처 완료")



=== onedayclass_topic 처리 전 (18종) ===
Top 10:
  데이터 시각화 (Matplotlib             350
  Seaborn 등)                      350
  머신러닝 / 딥러닝 응용                   315
  Python 응용                       265
  SQL 응용                          211
  웹 크롤링                           155
  테블로                               1
  태블로 학습                            1
  태블로                               1
  Power Bi 강연                       1
  ... +8개

=== onedayclass_topic 처리 후 (7종) ===
Top 10:
  데이터 시각화 (Matplotlib             350
  Seaborn 등)                      350
  머신러닝 / 딥러닝 응용                   315
  Python 응용                       265
  SQL 응용                          211
  웹 크롤링                           155
  희소                               12



✓ 7개 피처 완료


[desired_job_except_data] 

-> 멀티-핫 인코딩(다중 선택이 가능하므로)

In [856]:
# 9-4 desired_job_except_data 멀티-핫 인코딩 (최종)

col = 'desired_job_except_data'

# 0. 백업
orig_col = f'{col}_orig'
train_clean[orig_col] = train_clean[col]
valid_clean[orig_col] = valid_clean[col]
test_clean[orig_col] = test_clean[col]

# 1. 분할된 카테고리 확인
categories = []
for val in train_clean[orig_col].dropna():
    cats = [c.strip() for c in str(val).split(',') if c.strip()]
    categories.extend(cats)

cat_count = Counter(categories)
print(f"\n=== {col} 카테고리 ({len(cat_count)}종) ===")
for cat, cnt in sorted(cat_count.items(), key=lambda x: x[1], reverse=True):
    print(f"  {cat:50s} {cnt:4d}")

all_cats = sorted(cat_count.keys())
print(f"\n멀티-핫 피처: {len(all_cats)}개")

# 2. 멀티-핫 인코딩
new_cols = [f'{col}_{c}' for c in all_cats]
for df, orig in zip([train_clean, valid_clean, test_clean], 
                   [train_clean[orig_col], valid_clean[orig_col], test_clean[orig_col]]):
    def encode(val):
        if pd.isna(val): return pd.Series({c:0 for c in all_cats})
        cats = [c.strip() for c in str(val).split(',') if c.strip()]
        result = {c:0 for c in all_cats}
        for c in cats:
            if c in result: result[c] = 1
        return pd.Series(result)
    
    encoded_df = orig.apply(encode)
    df[new_cols] = encoded_df[all_cats].values

print(f"\n✓ {len(all_cats)}개 피처 완료")



=== desired_job_except_data 카테고리 (8종) ===
  B. 기획 / 전략 / 경영 직무                                  262
  A. 금융 / 보험 직무                                       220
  F. PM / 서비스 기획자                                     175
  D. 소프트웨어 개발자                                        171
  H. 마케터 / 영업관리                                       135
  G. 자연과학계열 연구자                                        92
  E. UI/UX 디자이너                                        69
  C. MD                                                42

멀티-핫 피처: 8개

✓ 8개 피처 완료


[expected_domain]

-> 

In [857]:
# expected_domain 알파벳 아닌 카테고리 '희소' 통합 (전체 출력)

col = 'expected_domain'

# 0. 백업
orig_col = f'{col}_orig'
train_clean[orig_col] = train_clean[col]
valid_clean[orig_col] = valid_clean[col]
test_clean[orig_col] = test_clean[col]

# 1. 처리 전 전체
categories = []
for val in train_clean[orig_col].dropna():
    cats = [c.strip() for c in str(val).split(',')]
    categories.extend(cats)

before_count = Counter(categories)
print(f"\n=== {col} 처리 전 ({len(before_count)}종) ===")
for cat, cnt in sorted(before_count.items(), key=lambda x: x[1], reverse=True):
    print(f"  {cat:40s} {cnt:4d}")

# 2. 알파벳 아닌 '희소' 통합
non_alpha_cats = [cat for cat in before_count if not re.match(r'^[A-Z]', cat)]
alpha_cats = [cat for cat in before_count if re.match(r'^[A-Z]', cat)]
after_cats = sorted(alpha_cats + ['희소'])

print(f"\n알파벳 아닌: {len(non_alpha_cats)}개 → '희소' 통합")
print(f"=== 처리 후 ({len(after_cats)}종) ===")
for cat in sorted(after_cats):
    if cat == '희소':
        cnt = sum(before_count[non_alpha] for non_alpha in non_alpha_cats)
    else:
        cnt = before_count[cat]
    print(f"  {cat:40s} {cnt:4d}")

# 3. 멀티-핫
new_cols = [f'{col}_{c}' for c in after_cats]
for df, orig in zip([train_clean, valid_clean, test_clean], 
                   [train_clean[orig_col], valid_clean[orig_col], test_clean[orig_col]]):
    def encode(val):
        if pd.isna(val): return pd.Series({c:0 for c in after_cats})
        cats = [c.strip() for c in str(val).split(',')]
        result = {c:0 for c in after_cats}
        for c in cats:
            if re.match(r'^[A-Z]', c) and c in result:
                result[c] = 1
            elif c != '':  # 빈 문자열 제외
                result['희소'] = 1
        return pd.Series(result)
    
    encoded_df = orig.apply(encode)
    df[new_cols] = encoded_df[after_cats].values

print(f"\n✓ {len(after_cats)}개 피처 완료")





=== expected_domain 처리 전 (30종) ===
  K. 금융 및 보험업                               221
  J. 정보통신업                                  215
  M. 전문                                     212
  과학 및 기술 서비스업                              212
  R. 예술                                      87
  스포츠 및 여가관련 서비스업                            87
  U. 국제 및 외국기관                               75
  C. 제조업                                     51
  O. 공공 행정                                   40
  국방 및 사회보장 행정                               40
  P. 교육 서비스업                                 32
  G. 도매 및 소매업                                27
  Q. 보건업 및 사회복지 서비스업                         26
  S. 협회 및 단체                                 16
  수리 및 기타 개인 서비스업                            16
  N. 사업시설 관리                                 14
  사업 지원 및 임대 서비스업                            14
  H. 운수 및 창고업                                11
  L. 부동산업                                    10
  I. 숙박 및 음식점업                               10
  D.

[desired_job] 

-> 데이터 수가 1인 희소 카테고리들을 모두 '희소'로 통합

그 후 멀티-핫 인코딩(다중 선택이 가능하므로)

In [858]:
# desired_job 희소(1개) → '희소' 통합 (전체 출력)

col = 'desired_job'

# 0. 백업
orig_col = f'{col}_orig'
train_clean[orig_col] = train_clean[col]
valid_clean[orig_col] = valid_clean[col]
test_clean[orig_col] = test_clean[col]

# 1. 처리 전 전체
categories = []
for val in train_clean[orig_col].dropna():
    cats = [c.strip() for c in str(val).split(',') if c.strip()]
    categories.extend(cats)

before_count = Counter(categories)
print(f"\n=== {col} 처리 전 ({len(before_count)}종) ===")
for cat, cnt in sorted(before_count.items(), key=lambda x: x[1], reverse=True):
    print(f"  {repr(cat):50s} {cnt:4d}")

# 2. 희소 통합
rare_cats = {cat for cat, cnt in before_count.items() if cnt == 1}
after_cats = sorted((set(before_count) - rare_cats) | {'희소'})

print(f"\n희소(1개): {len(rare_cats)}개")
print(f"=== 처리 후 ({len(after_cats)}종) ===")
희소_cnt = sum(before_count[rc] for rc in rare_cats)
print(f"  희소{'':>30s} {희소_cnt:4d}")

for cat in sorted(c for c in after_cats if c != '희소'):
    print(f"  {cat:50s} {before_count[cat]:4d}")

# 3. 멀티-핫
new_cols = [f'{col}_{c}' for c in after_cats]
for df, orig in zip([train_clean, valid_clean, test_clean], 
                   [train_clean[orig_col], valid_clean[orig_col], test_clean[orig_col]]):
    def encode(val):
        if pd.isna(val): return pd.Series({c:0 for c in after_cats})
        cats = [c.strip() for c in str(val).split(',') if c.strip()]
        result = {c:0 for c in after_cats}
        for c in cats:
            key = '희소' if c in rare_cats else c
            result[key] = 1
        return pd.Series(result)
    
    df[new_cols] = orig.apply(encode)[after_cats].values

print(f"\n✓ {len(after_cats)}개 피처 완료")



=== desired_job 처리 전 (62종) ===
  'B. 데이터 분석가'                                        386
  'C. 데이터 사이언티스트'                                     241
  'A. 데이터 엔지니어'                                       150
  'D. 인공지능 전문가'                                        95
  'I. 마케터'                                             89
  'G. PM/서비스 기획자'                                      80
  'E. 소프트웨어 개발자'                                       59
  'H. 자연과학계열 연구자'                                      28
  'F. UI/UX 디자이너'                                      18
  'J. MD'                                              17
  '반도체 엔지니어'                                            5
  '산림빅데이터 전문가'                                          1
  '반도체 공정'                                              1
  '없음'                                                  1
  '인사팀'                                                 1
  '픔질관리'                                                1
  'ESG 경영/공시 직무'                        

[major_field]

1. 'IT (컴퓨터 공학 포함)', '공학 (컴퓨터 공학 제외)', '자연과학', '자연고학' 를 '이공계'로 통합

2. '경영학', '사회과학', '인문학', '경제통상학', '교육학', '법학' 를 '인문계'로 통합 

3. '예체능', '의약학', 'Unknown'은 그대로 유지.

-> [major1_1] 및 [major1_2]에 이미 세부 전공 정보가 있는데 [major_field]까지 '경영학', '경제통상학' 등을 다룰 필요가 없다고 판단.

그 후 멀티-핫 인코딩(다중 선택이 가능하므로)

In [859]:
# 9-7 셀: major_field 그룹화 + 멀티-핫

col = 'major_field'

# 0. 백업
orig_col = f'{col}_orig'
train_clean[orig_col] = train_clean[col]
valid_clean[orig_col] = valid_clean[col]
test_clean[orig_col] = test_clean[col]

# 1. 처리 전
categories = []
for val in train_clean[orig_col].dropna():
    cats = [c.strip() for c in str(val).split(',')]
    categories.extend(cats)

before_count = Counter(categories)
print(f"\n=== {col} 처리 전 ({len(before_count)}종) ===")
for cat, cnt in sorted(before_count.items(), key=lambda x: x[1], reverse=True):
    print(f"  {cat:40s} {cnt:4d}")

# 2. 그룹화 매핑
def group_major(cat):
    eng = ['IT (컴퓨터 공학 포함)', '공학 (컴퓨터 공학 제외)', '자연과학', '자연고학']
    hum = ['경영학', '사회과학', '인문학', '경제통상학', '교육학', '법학']
    
    if cat in eng: return '이공계'
    elif cat in hum: return '인문계'
    elif cat in ['예체능', '의약학', 'Unknown']: return cat
    else: return '기타'  # 혹시 모를 나머지

# 3. 처리 후 확인
after_categories = [group_major(cat) for cat in categories]
after_count = Counter(after_categories)
after_cats = sorted(set(after_categories))

print(f"\n=== {col} 처리 후 ({len(after_cats)}종) ===")
for cat, cnt in sorted(after_count.items(), key=lambda x: x[1], reverse=True):
    print(f"  {cat:40s} {cnt:4d}")

# 4. 멀티-핫 인코딩
new_cols = [f'{col}_{c}' for c in after_cats]
for df, orig in zip([train_clean, valid_clean, test_clean], 
                   [train_clean[orig_col], valid_clean[orig_col], test_clean[orig_col]]):
    def encode(val):
        if pd.isna(val): return pd.Series({c:0 for c in after_cats})
        cats = [c.strip() for c in str(val).split(',')]
        result = {c:0 for c in after_cats}
        for c in cats:
            grouped = group_major(c)
            if grouped in result: result[grouped] = 1
        return pd.Series(result)
    
    encoded_df = orig.apply(encode)
    df[new_cols] = encoded_df[after_cats].values

print(f"\n✓ {len(after_cats)}개 피처 완료")



=== major_field 처리 전 (13종) ===
  IT (컴퓨터 공학 포함)                            151
  공학 (컴퓨터 공학 제외)                            126
  경영학                                       114
  자연과학                                       89
  사회과학                                       83
  인문학                                        51
  경제통상학                                      29
  Unknown                                    18
  예체능                                        13
  의약학                                         8
  교육학                                         6
  법학                                          3
  자연고학                                        1

=== major_field 처리 후 (5종) ===
  이공계                                       367
  인문계                                       286
  Unknown                                    18
  예체능                                        13
  의약학                                         8

✓ 5개 피처 완료
